# Geometry-V1 Batch 2B
PREPARED_NOT_EXECUTED; transport-only; science_denominator=0.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
import json,os,pathlib,shutil,subprocess,sys,re
import numpy as np
from PIL import Image
from google.colab import userdata
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V1'; EXECUTION_EXACT='621a33310493356c0bc295c9122a5888dfcae501'; RUN_ID='geometry-v1-b2b-621a33310493-operational-01'
PROPOSED_PENDING_FINAL_USER_CONFIRMATION='/content/drive/MyDrive/CEG-WM/Geometry-V1/Batch2B'; MAX_CONTROL_BYTES=1024; MAX_RECEIPT_BYTES=262144; MAX_ARCHIVE_BYTES=524288; MAX_SIDECAR_BYTES=256
DRIVE_ROOT=pathlib.Path(PROPOSED_PENDING_FINAL_USER_CONFIRMATION); repo=pathlib.Path('/content/geometry-v1-source'); input_dir=pathlib.Path('/content/geometry-v1-inputs'); output_root=pathlib.Path('/content/geometry-v1-runner-output'); run_dir=DRIVE_ROOT/RUN_ID
root_key=''; hf_token=''; runner_env=None; process=None; control_read=None; control_write=None; input_paths=[]; fixed_image=None; fixed_array=None; created_run_dir=False
def build_fixed_operational_rgb() -> Image.Image:
 yy,xx=np.indices((512,512),dtype=np.uint16); array=np.stack(((3*xx+5*yy)%256,(7*xx+2*yy+17*(xx//32))%256,(xx^(3*yy))%256),axis=-1).astype(np.uint8); array[40:180,55:225]=(241,67,31); array[300:465,335:493]=(19,181,223); return Image.fromarray(array)
def verify_checkout():
 h=subprocess.run(['git','rev-parse','HEAD'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip(); c=subprocess.run(['git','status','--porcelain'],cwd=repo,check=True,capture_output=True,text=True).stdout.strip()
 if h!=EXECUTION_EXACT or c: raise RuntimeError('checkout identity differs')
def parse_child(rc):
 line=os.read(control_read,MAX_CONTROL_BYTES+1)
 if len(line)>MAX_CONTROL_BYTES: raise RuntimeError('bounded control exceeded')
 text=line.decode('utf-8','strict'); success='CEGWM_GEOMETRY_V1_OPERATIONAL_PREFLIGHT '; failure='CEGWM_GEOMETRY_V1_OPERATIONAL_FAILURE '
 if not text.endswith('\n') or len(text.splitlines())!=1: raise RuntimeError('exactly one control line required')
 if text.startswith(success): prefix,status=success,'success'
 elif text.startswith(failure): prefix,status=failure,'failure'
 else: raise RuntimeError('invalid control prefix')
 payload=json.loads(text[len(prefix):]); base={'status','run_id','artifact_status','archive_filename','sidecar_filename','receipt_bytes','receipt_sha256','archive_bytes'}; allowed=base if status=='success' else base|{'underlying_status','failure_point'}
 if set(payload)!=allowed or payload['run_id']!=RUN_ID or (rc==0)!=(status=='success') or (status=='success')!=(payload['artifact_status']=='complete'): raise RuntimeError('control/return-code mismatch')
 return status,payload
def exclusive_copy(source,target):
 with source.open('rb') as read,target.open('xb') as write:
  while chunk:=read.read(1048576): write.write(chunk)
try:
 if any(p.exists() for p in (repo,input_dir,output_root,run_dir)): raise FileExistsError('create-only path exists')
 subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); subprocess.run(['git','checkout','--detach',EXECUTION_EXACT],cwd=repo,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout(); subprocess.run([sys.executable,'-m','pip','install',str(repo)],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); verify_checkout()
 input_dir.mkdir(); fixed_image=build_fixed_operational_rgb(); fixed_array=np.asarray(fixed_image); fixed_path=input_dir/'geometry_v1_batch2b_fixed_rgb.png'; input_paths.append(fixed_path)
 if fixed_image.mode!='RGB' or fixed_image.size!=(512,512) or fixed_array.dtype!=np.uint8 or fixed_array.shape!=(512,512,3): raise RuntimeError('fixed RGB input identity differs')
 with fixed_path.open('xb') as handle: fixed_image.save(handle,format='PNG')
 hf_token=userdata.get('HF_TOKEN'); root_key=userdata.get('CEG_WM_ROOT_KEY');
 if not hf_token or not root_key: raise RuntimeError('required secret unavailable')
 runner_env={n:v for n,v in os.environ.items() if all(x not in n.upper() for x in ('TOKEN','KEY','SECRET'))}; runner_env['HF_TOKEN']=hf_token; runner_env['CEG_WM_ROOT_KEY']=root_key; control_read,control_write=os.pipe()
 command=[sys.executable,'-m','experiments.run_geometry_v1_qk_operational_preflight','--repo-root',str(repo),'--expected-exact',EXECUTION_EXACT,'--output-root',str(output_root),'--control-fd',str(control_write),str(fixed_path)]
 process=subprocess.Popen(command,cwd=repo,env=runner_env,pass_fds=(control_write,),stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL); os.close(control_write); control_write=None; process.wait(timeout=1800); status,payload=parse_child(process.returncode)
 archive=output_root/payload['archive_filename']; sidecar=output_root/payload['sidecar_filename']; sidecar_value=sidecar.read_bytes()
 if not archive.is_file() or not sidecar.is_file() or archive.stat().st_size!=payload['archive_bytes'] or archive.stat().st_size>MAX_ARCHIVE_BYTES or payload['receipt_bytes']>MAX_RECEIPT_BYTES or len(sidecar_value)>MAX_SIDECAR_BYTES or re.fullmatch(rb'[0-9a-f]{64}  '+re.escape(archive.name.encode())+rb'\n',sidecar_value) is None: raise RuntimeError('runner package declaration differs')
 DRIVE_ROOT.mkdir(parents=True,exist_ok=True); run_dir.mkdir(); created_run_dir=True; exclusive_copy(archive,run_dir/archive.name); exclusive_copy(sidecar,run_dir/sidecar.name)
 if status=='failure': raise RuntimeError('child reported packaged operational failure')
finally:
 root_key=''; hf_token=''; fixed_image=None; fixed_array=None
 if runner_env is not None: runner_env.pop('HF_TOKEN',None); runner_env.pop('CEG_WM_ROOT_KEY',None)
 if process is not None and process.poll() is None: process.kill(); process.wait()
 for fd in (control_read,control_write):
  if fd is not None: os.close(fd)
 for path in input_paths:
  if path.exists(): path.unlink()
 if input_dir.exists(): input_dir.rmdir()
 if output_root.exists(): shutil.rmtree(output_root)
 if repo.exists(): shutil.rmtree(repo)
 if created_run_dir and run_dir.exists() and len(list(run_dir.iterdir()))!=2: shutil.rmtree(run_dir)
